# Validation for BERTopic
Drawing on the validation framework proposed by Maier and colleagues, this study utilized model stability comparison, C_v, manual topic review, and Diversity to validate the results in terms of reliability, interpretability, and validity. Considering the size of the corpus, this study used a 25% sample of the original data for validation.

In [1]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
sys.path.append(os.path.abspath(".."))
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import itertools
from bertopic.vectorizers import ClassTfidfTransformer
import scipy.sparse as sp
# NPMI
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

f:\Python\Setting\Miniconda\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")

True
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
embeddings = np.load('../data/layer2/embeddings/emb.npy')
windows = pd.read_csv("../data/layer2/layer2_window.csv")
states = pd.read_csv('../data/preparation/states.csv')

docs = windows['window_content'].astype(str).tolist()
docs = list(dict.fromkeys(docs))


sample_frac = 0.25
sample_n = int(len(docs) * sample_frac)
sample_idx = np.random.default_rng(42).choice(len(docs), size=sample_n, replace=False)

docs_sample = [docs[i] for i in sample_idx]
embeddings_sample = embeddings[sample_idx]

print(f"Original dataset: {len(docs):,} documents")
print(f"Sample size (25%): {len(docs_sample):,} documents")
print(f"Sample embeddings shape: {embeddings_sample.shape}")

Original dataset: 125,717 documents
Sample size (25%): 31,429 documents
Sample embeddings shape: (31429, 384)


In [4]:
embedding_model = SentenceTransformer('F:/Python/Projects/Project2025/GroupA/models/all-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6239.66it/s]
BertModel LOAD REPORT from: F:/Python/Projects/Project2025/GroupA/models/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Build robust custom stop words from states.csv (drop NaN/numeric noise)
custom_words = (
    states.select_dtypes(include=['object', 'string'])
    .stack()
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)
custom_words = [w for w in custom_words if w]
customized_stop_words = list(text.ENGLISH_STOP_WORDS.union(custom_words))
print(f"Custom stop words loaded: {len(customized_stop_words):,}")

Custom stop words loaded: 505


## Model Stability


The study repeatedly ran the optimized model on the sample using five different random states. Therefore, it obtained 10 pairwise comparisons of results. It then calculated similarity matrices each time based on the top 10 words between topics from the two results. Similarity was calculated as follows:
For two topics a and b from runs X and Y, respectively:

$$\text{Similarity}(T_{Xa}, T_{Yb}) = \cos(\theta) = \frac{\mathbf{v}_{Xa} \cdot \mathbf{v}_{Yb}}{\|\mathbf{v}_{Xa}\| \|\mathbf{v}_{Yb}\|}$$

Then, the study calculated similarities between all topics from run X and run Y, and for each topic in run X (or run Y) selected the largest similarity above 0.7 as the best pair. The study then calculated the ratio of the number of best pairs to the larger total number of topics in run X and run Y. This ratio is defined as the reliability score.

$$\text{Reliability Score}_{X,Y} = \frac{| \{ T \in \text{Set}_X : \max_{T' \in \text{Set}_Y} \text{Sim}(T, T') \ge 0.7 \} |}{\max(N_X, N_Y)}$$

The study obtained confidence intervals from the 10 reliability scores generated by mutually comparing the results of the five models. This study reports an acceptable mean reliability score of 0.601 based on five independent experiments, indicating a moderate level of reproducibility.

https://elibrary.utb.de/doi/book/10.1453/9783869623870

In [6]:
best_params = {
    'n_neighbors': 120,
    'n_components': 10,
    'min_dist': 0.033068211866161414,
    'min_cluster_size': 10,
    'min_samples': 2,
    'cluster_selection_epsilon': 0.14993193586957335,
    'min_topic_size': 30
}

vectorizer_model = CountVectorizer(
    stop_words=customized_stop_words,
    ngram_range=(1, 2),
)

vectorizer_model.fit(docs_sample)

hdbscan_model = HDBSCAN(
    min_cluster_size=best_params['min_cluster_size'],
    min_samples=best_params['min_samples'],
    metric='euclidean',
    cluster_selection_epsilon=best_params['cluster_selection_epsilon'],
    prediction_data=True
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=False) 

In [7]:
seeds = [11, 22, 33, 44, 55]
models = []

for i, seed in enumerate(seeds):
    print(f"Run {i+1}/5 (Seed: {seed})...")
    
    umap_model = UMAP(
        n_neighbors=best_params['n_neighbors'],
        n_components=best_params['n_components'],
        min_dist=best_params['min_dist'],
        metric='cosine',
        random_state=seed
    )
    
    model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=best_params['min_topic_size'],
        nr_topics='auto'
    )
    
    model.fit_transform(docs_sample)
    models.append(model)

Run 1/5 (Seed: 11)...
Run 2/5 (Seed: 22)...
Run 3/5 (Seed: 33)...
Run 4/5 (Seed: 44)...
Run 5/5 (Seed: 55)...


In [31]:
def get_topic_dict(model, topic_id, top_k=10):
    if topic_id == -1: return None
    return {word: weight for word, weight in model.get_topic(topic_id)[:top_k] if word != ''}

def calc_dict_cosine(dict1, dict2):
    if dict1 is None or dict2 is None: return 0
    vocab = list(set(dict1.keys()) | set(dict2.keys()))

    v1 = np.array([dict1.get(word, 0) for word in vocab]).reshape(1, -1)
    v2 = np.array([dict2.get(word, 0) for word in vocab]).reshape(1, -1)
    
    return cosine_similarity(v1, v2)[0][0]

In [32]:
reliability_scores = []
pairs = list(itertools.combinations(range(len(seeds)), 2))

for i, j in pairs:
    model_i = models[i]
    model_j = models[j]
    
    topics_i = [t for t in model_i.get_topic_info()['Topic'] if t != -1]
    topics_j = [t for t in model_j.get_topic_info()['Topic'] if t != -1]
    
    num_i = len(topics_i)
    num_j = len(topics_j)
    
    if num_i == 0 or num_j == 0:
        reliability_scores.append(0)
        continue

    sim_matrix = np.zeros((num_i, num_j))
    
    for r_idx, t_i in enumerate(topics_i):
        dict_i = get_topic_dict(model_i, t_i, top_k=10)
        for c_idx, t_j in enumerate(topics_j):
            dict_j = get_topic_dict(model_j, t_j, top_k=10)
            sim_matrix[r_idx, c_idx] = calc_dict_cosine(dict_i, dict_j)
 
    best_matches = sim_matrix.max(axis=1)
    reliability_count = (best_matches >= 0.7).sum()
    ratio = reliability_count / max(num_i, num_j)
    
    reliability_scores.append(ratio)
    print(f"Pair ({i}, {j}) reliability_score: {ratio:.4f}")

Pair (0, 1) reliability_score: 0.6316
Pair (0, 2) reliability_score: 0.5088
Pair (0, 3) reliability_score: 0.7193
Pair (0, 4) reliability_score: 0.6667
Pair (1, 2) reliability_score: 0.6042
Pair (1, 3) reliability_score: 0.6296
Pair (1, 4) reliability_score: 0.6250
Pair (2, 3) reliability_score: 0.5185
Pair (2, 4) reliability_score: 0.4821
Pair (3, 4) reliability_score: 0.6250


In [ ]:
mean_reliability = np.mean(reliability_scores)
std_err = stats.sem(reliability_scores) 
confidence = 0.95
# t-distribution for small-size sample
h = std_err * stats.t.ppf((1 + confidence) / 2., len(reliability_scores) - 1)

print(f"Mean Stability: {mean_reliability:.3f}")
print(f"Std Deviation: {np.std(reliability_scores):.3f}")
print(f"95% CI: [{mean_reliability - h:.3f}, {mean_reliability + h:.3f}]")

Mean Stability: 0.601
Std Deviation: 0.071
95% CI: [0.547, 0.655]


: 

# Interpretability
According to prior studies, intrinsic coherence is correlated with human interpretability of topics (Mimno et al., 2011). The two metrics below are widely used in current scholarship.
## NPMI
NPMI (Normalized Pointwise Mutual Information) estimates the likelihood that two words co-occur compared with the likelihood that they co-occur by chance. The ratio is then normalized by its upper bound (when PMI is maximal, i.e., when words i and j always appear together). This metric is significant for measuring coherence.

$$PMI(w_i, w_j) = \log \frac{P(w_i, w_j)}{P(w_i)P(w_j)}$$
Therefore, 
$$PMI_{max} = \log \frac{P(w_i)}{P(w_i)P(w_i)} = \log \frac{1}{P(w_i)} = -\log P(w_i) \text{ or } -\log P(w_i, w_j)$$
$$NPMI(w_i, w_j) = \frac{PMI(w_i, w_j)}{-\log P(w_i, w_j)}$$

In this study, the NPMI is very low (0.051) due to the window size. Compared with full news articles, three-sentence windows provide limited space for two words to co-occur. Meanwhile, NPMI ignores semantic distances between words, which may further reduce intrinsic coherence.

https://www.semanticscholar.org/paper/Normalized-(pointwise)-mutual-information-in-Bouma/15218d9c029cbb903ae7c729b2c644c24994c201

In [28]:
# tokenization
analyzer = vectorizer_model.build_analyzer()
tokenized_docs = [analyzer(doc) for doc in docs_sample]
dictionary = Dictionary(tokenized_docs)

all_run_npmi = []

for i, model in enumerate(models):
    topic_words_list = []
    
    topics = [t for t in model.get_topic_info()['Topic'] if t != -1]
    
    for t in topics:
        # extract the top 10 words
        raw_words = [word for word, _ in model.get_topic(t)[:10]]
        
        # fileter out the word that is not in the dic (nan)
        valid_words = [w for w in raw_words if w in dictionary.token2id]
        
        # NPMI can only be calculated when the words number is 2 or onwards
        if len(valid_words) > 1:
            topic_words_list.append(valid_words)

    if not topic_words_list:
        print(f"Run {i}: not valid, skip")
        all_run_npmi.append(np.nan)
        continue
        
    # Genism
    cm = CoherenceModel(
        topics=topic_words_list, 
        texts=tokenized_docs, 
        dictionary=dictionary, 
        coherence='c_npmi'
    )
    
    score = cm.get_coherence()
    all_run_npmi.append(score)
    print(f"Run {i} NPMI (Fixed): {score:.3f}")

final_mean = np.nanmean(all_run_npmi)
print(f"\n Mean NPMI: {final_mean:.3f}")

Run 0 NPMI (Fixed): 0.050
Run 1 NPMI (Fixed): 0.057
Run 2 NPMI (Fixed): 0.059
Run 3 NPMI (Fixed): 0.053
Run 4 NPMI (Fixed): 0.038

 Mean NPMI: 0.051


## $C_v$
Based on these limitations of NPMI, the study utilized $C_v$ (vector-based coherence) to incorporate semantic distances. For each word $w_i$ in a topic, the metric first constructs a context vector by calculating the word's NPMI values with other words in the same topic.

$$\vec{v}_i = [NPMI(w_i, w_1), NPMI(w_i, w_2), \dots, NPMI(w_i, w_n)]$$

It then calculates the cosine similarity between the context vectors of two words ($w_i$ and $w_j$).

$$\phi(w_i, w_j) = \frac{\vec{v}_i \cdot \vec{v}_j}{\|\vec{v}_i\| \|\vec{v}_j\|}$$

Finally, the study computes the arithmetic mean of all similarities.

$$C_v = \frac{1}{N} \sum_{i} \phi(w_i, W)$$

In this study, the mean $C_v$ across five random experiments is 0.694, indicating strong coherence within topics and, therefore, high interpretability for human evaluation.

https://svn.aksw.org/papers/2015/WSDM_Topic_Evaluation/public.pdf

In [29]:
# tokenization
analyzer = vectorizer_model.build_analyzer()
tokenized_docs = [analyzer(doc) for doc in docs_sample]
dictionary = Dictionary(tokenized_docs)

all_run_cv = []

for i, model in enumerate(models):
    topic_words_list = []
    
    topics = [t for t in model.get_topic_info()['Topic'] if t != -1]
    
    for t in topics:
        # extract the top 10 words
        raw_words = [word for word, _ in model.get_topic(t)[:10]]
        
        # fileter out the word that is not in the dic (nan)
        valid_words = [w for w in raw_words if w in dictionary.token2id]
        
        # NPMI can only be calculated when the words number is 2 or onwards
        if len(valid_words) > 1:
            topic_words_list.append(valid_words)

    if not topic_words_list:
        print(f"Run {i}: not valid, skip")
        all_run_cv.append(np.nan)
        continue
        
    # Genism
    cm = CoherenceModel(
        topics=topic_words_list, 
        texts=tokenized_docs, 
        dictionary=dictionary, 
        coherence='c_v'
    )
    
    score = cm.get_coherence()
    all_run_cv.append(score)
    print(f"Run {i} C_v (Fixed): {score:.3f}")

final_mean = np.nanmean(all_run_cv)
print(f"\n Mean C_v: {final_mean:.3f}")

Run 0 C_v (Fixed): 0.696
Run 1 C_v (Fixed): 0.701
Run 2 C_v (Fixed): 0.715
Run 3 C_v (Fixed): 0.680
Run 4 C_v (Fixed): 0.680

 Mean C_v: 0.694


# Diversity
This metric was proposed by Dieng and colleagues and is measured as the proportion of unique words among the top 25 words across topics. The closer the value is to 1, the more diverse the topics are. In this study, the mean diversity is 0.898, indicating clear distinctions between topics and contributing to a high inter-topic validity.

However, this metric is also limited because it ignores semantic distances. In future studies, I would like to improve it by drawing an analogy to $C_v$.

https://aclanthology.org/2020.tacl-1.29/

In [30]:
def get_diversity(model):
    topics = [t for t in model.get_topic_info()['Topic'] if t != -1]
    all_words = []
    for t in topics:
        words = [w for w, _ in model.get_topic(t)[:25]]
        all_words.extend(words)
    
    if not all_words: return 0
    return len(set(all_words)) / len(all_words)

diversities = [get_diversity(m) for m in models]
print(f"Diversity: { [round(d, 4) for d in diversities] }")
print(f"Mean Topic Diversity: {np.mean(diversities):.3f}")

Diversity: [0.8965, 0.8938, 0.9225, 0.9111, 0.8643]
Mean Topic Diversity: 0.898
